In [1]:
import json
import os
import sys
import time
import copy
from pathlib import Path
from typing import Iterable, List, Tuple, Dict, Union, Optional
import warnings 
import argparse

import matplotlib
import numpy as np
import pandas as pd
import torch
import wandb
from scipy.sparse import csr_matrix
from torch import nn
from torch.nn import functional as F
from torchtext.vocab import Vocab
from torchtext._torchtext import Vocab as VocabPybind
from torch_geometric.loader import DataLoader
from torch.utils.data import ConcatDataset
from gears import PertData, GEARS
from gears.inference import compute_metrics, deeper_analysis, non_dropout_analysis
from gears.utils import create_cell_graph_dataset_for_prediction

PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon")
sys.path.insert(0, os.path.join(PATH, "scripts/scGPT"))

import scgpt as scg
from scgpt.model import TransformerGenerator
from scgpt.loss import masked_mse_loss, criterion_neg_log_bernoulli, masked_relative_error
from scgpt.tokenizer import tokenize_batch, pad_batch, tokenize_and_pad_batch
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.utils import set_seed, map_raw_id_to_vocab_id, compute_perturbation_metrics

/usr3/graduate/andrewdr/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pert_data_source = PertData(os.path.join(PATH, "data/scgpt"))
pert_data_target = PertData(os.path.join(PATH, "data/scgpt"))
pert_data_source.load(data_path = os.path.join(PATH, "data/scgpt/rpe1_shared"))
pert_data_target.load(data_path = os.path.join(PATH, "data/scgpt/k562_shared"))

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['C14orf178+ctrl' 'C7orf26+ctrl' 'MTRNR2L1+ctrl' 'RBM14-RBM4+ctrl'
 'GNB1L+ctrl' 'ALG1L+ctrl' 'FAU+ctrl' 'CCDC144NL+ctrl' 'GOLGA6L1+ctrl'
 'RPS10-NUDT3+ctrl' 'FAM229A+ctrl' 'FAM102B+ctrl' 'C19orf53+ctrl'
 'SEM1+ctrl' 'KRTAP4-7+ctrl' 'C16orf86+ctrl' 'AC118549.1+ctrl'
 'NEDD8-MDP1+ctrl' 'OR4F4+ctrl' 'C18orf21+ctrl']
Local copy of pyg dataset is detected. Loading...
Done!
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['C7orf26+ctrl' 'C19orf53+ctrl' 'KRTAP4-7+ctrl' 'OR4F4+ctrl'
 'C16orf86+ctrl' 'CCDC144NL+ctrl' 'ALG1L+ctrl' 'GOLGA6L1+ctrl'
 'C14orf178+ctrl' 'RPS10-NUDT3+ctrl' 'FAM229A+ctrl' 'SEM1+ctrl'
 'GNB1L+ctrl' 'MTRNR2L1+ctrl' 'FAU+ctrl']
Local copy of pyg dataset is detected. Loading...
Done!


In [34]:
pert_data_source.prepare_split(split = "no_split", seed = 42) 
pert_data_source_dataloader = pert_data_source.get_dataloader(batch_size=64, test_batch_size=64)
pert_data_target.prepare_split(split = "no_split", seed = 42) 
pert_data_target_dataloader=pert_data_target.get_dataloader(batch_size=64, test_batch_size=64)

Local copy of split is detected. Loading...
Done!
Creating dataloaders....
Dataloaders created...
Local copy of split is detected. Loading...
Done!
Creating dataloaders....
Dataloaders created...


In [23]:
target_pbs = set(pert_data_target.pert_names)
source_pbs = set(pert_data_source.pert_names)

In [22]:
pb_set = set(splits_df.pb)
splits_df = pd.read_csv(os.path.join(PATH, "data/sigs/perturb-seq/pb_splits.csv")) # splits df is all in rpe1_shared and k562_shared

In [27]:
target_pbs.issubset(source_pbs), source_pbs.issubset(target_pbs)

(True, True)

In [28]:
pb_set.issubset(source_pbs)

False

In [30]:
pb_set - target_pbs

{'ALG1L',
 'C14orf178',
 'C16orf86',
 'C19orf53',
 'C7orf26',
 'CCDC144NL',
 'FAM229A',
 'FAU',
 'GNB1L',
 'GOLGA6L1',
 'KRTAP4-7',
 'MTRNR2L1',
 'OR4F4',
 'RPS10-NUDT3',
 'SEM1'}

In [29]:
pb_set - source_pbs

{'ALG1L',
 'C14orf178',
 'C16orf86',
 'C19orf53',
 'C7orf26',
 'CCDC144NL',
 'FAM229A',
 'FAU',
 'GNB1L',
 'GOLGA6L1',
 'KRTAP4-7',
 'MTRNR2L1',
 'OR4F4',
 'RPS10-NUDT3',
 'SEM1'}

In [4]:
unique_perturbations = np.unique([data["pert_idx"][0] for data in pert_data_target_dataloader["test_loader"].dataset])

In [11]:
unique_perturbations

2042

In [17]:
pert_data_target_dataloader["test_loader"].dataset[1]

Data(x=[7226, 1], y=[1, 7226], pert_idx=[1], de_idx=[20], pert='NAF1+ctrl')

In [39]:
pert_data_target_dataloader["test_loader"].dataset[1].pert.replace("+ctrl", "")

'NAF1'

In [19]:
pert_data_target.pert_names[pert_data_target_dataloader["test_loader"].dataset[1].pert_idx]

array(['NAF1'], dtype='<U10')

In [40]:
split_num = 1
# Get unique perturbations from the dataset
unique_perturbations = np.unique([data["pert_idx"][0] for data in pert_data_target_dataloader["test_loader"].dataset])

# Extract perturbation names (assuming data has 'pert' field with gene names)
# You may need to map pert_idx back to gene names depending on your data structure
pert_idx_to_name = {data["pert_idx"][0]: data["pert"].replace("+ctrl","") for data in pert_data_target_dataloader["test_loader"].dataset}
pert_name_to_idx = {v: k for k, v in pert_idx_to_name.items()}

# Get the split column name
split_col = f"split_{split_num}"

# Get perturbations in the eval split (TRUE values = held out 1/10th)
eval_pb_names = splits_df[splits_df[split_col] == True]["pb"].values
train_pb_names = splits_df[splits_df[split_col] == False]["pb"].values

print(f"Using split {split_num}: {len(eval_pb_names)} eval perturbations, {len(train_pb_names)} train perturbations")

# Convert names to indices (filter to only those present in the dataset)
eval_perturbations = [pert_name_to_idx[name] for name in eval_pb_names if name in pert_name_to_idx]
train_perturbations = [pert_name_to_idx[name] for name in train_pb_names if name in pert_name_to_idx]

print(f"Found {len(eval_perturbations)} eval and {len(train_perturbations)} train perturbations in dataset")

# Split the target dataset based on predefined splits
train_data_split = [data for data in pert_data_target_dataloader["test_loader"].dataset 
                    if data["pert_idx"][0] in train_perturbations]
eval_data_split = [data for data in pert_data_target_dataloader["test_loader"].dataset 
                   if data["pert_idx"][0] in eval_perturbations]

# Concatenate the datasets (All source, 90% target)
combined_dataset = ConcatDataset([pert_data_source_dataloader["test_loader"].dataset, train_data_split])

# Create DataLoaders
combined_train_dataloader = DataLoader(combined_dataset, batch_size=64, shuffle=True)
eval_dataloader = DataLoader(eval_data_split, batch_size=64, shuffle=True)

print(f"Train dataloader size: {len(combined_train_dataloader.dataset)}")
print(f"Eval dataloader size: {len(eval_dataloader.dataset)}")

Using split 1: 206 eval perturbations, 1847 train perturbations
Found 204 eval and 1834 train perturbations in dataset
Train dataloader size: 501848
Eval dataloader size: 29184


In [43]:
eval_perturbations

[39,
 103,
 196,
 245,
 243,
 454,
 470,
 487,
 631,
 639,
 837,
 847,
 931,
 988,
 993,
 1043,
 1103,
 1111,
 1133,
 1222,
 1200,
 1242,
 1251,
 1283,
 1354,
 1385,
 1546,
 1588,
 1613,
 1640,
 1688,
 1733,
 1752,
 1813,
 1834,
 1898,
 1899,
 2082,
 2121,
 2141,
 2286,
 2287,
 2296,
 2317,
 2321,
 2364,
 2435,
 2518,
 2523,
 2521,
 2549,
 2594,
 2645,
 2700,
 2806,
 2823,
 3006,
 3037,
 3055,
 3076,
 3098,
 3099,
 3129,
 3173,
 3199,
 3227,
 3384,
 3385,
 3404,
 3438,
 3470,
 3619,
 3715,
 3724,
 3852,
 3890,
 3916,
 4004,
 4016,
 4115,
 4252,
 4366,
 4394,
 4454,
 4430,
 4435,
 4452,
 4546,
 4569,
 4579,
 4686,
 4713,
 4745,
 4748,
 4720,
 4727,
 4736,
 4773,
 4795,
 4926,
 4930,
 4931,
 4993,
 5001,
 5014,
 5018,
 5058,
 5115,
 5165,
 5174,
 5190,
 5251,
 5252,
 5313,
 5347,
 5350,
 5352,
 5378,
 5392,
 5442,
 5464,
 5482,
 5489,
 5616,
 5658,
 5697,
 5732,
 5804,
 5896,
 5956,
 6008,
 6012,
 6030,
 6044,
 6079,
 6212,
 6214,
 6221,
 6225,
 6232,
 6249,
 6251,
 6427,
 6506,
 6529,
 